# Synthetic preprocessing 00 — Combine base-model and fine-tuned LoRA outputs

Use this notebook **after generation and human review**. Attach each downloaded Kaggle generation run as a Kaggle Input. It combines reviewed CSV metadata only; generated image bytes remain in their original input mounts. It verifies original SHA-256 hashes, file existence, original prompt family, actual reviewer approval, and consistent generator mode/weights per class. **Do not blindly merge base, textual inversion, and custom LoRA image pools into one research arm**. To compare those generators, combine them in separate runs.

**Important:** The original `generation_manifest.csv` marks every image as `accepted=False` / `reviewer_decision=needs_review`. Complete and save a separately reviewed copy, including `reviewer_id`, before enabling the combine step. Synthetic appearances are not medical diagnoses.

The output `combined_reviewed_synthetic_pool.csv` is an input to `DermaLens_Synthetic_Prompt_Split_Kaggle.ipynb`; it preserves `prompt_partition`, `prompt_family_id`, `prompt_type`, generator checkpoint and adapter metadata. Full train/val/test synthetic splitting requires images from all three prompt partitions; a train-only pilot should stay train-only until separately generated val/test families exist.

**Mixed base + LoRA support:** `ALLOW_MIXED_GENERATORS=True` combines reviewed sources into one preprocessing CSV while preserving every row's generator mode and adapter revision. It does not mean mixed outputs are a valid controlled E1/E2 arm. Filter the resulting *preprocessed* pool by `--generator-mode` in `scripts/build_experiment_manifests.py` before classifier comparisons.


In [ ]:
from pathlib import Path
import sys,subprocess,json
REPO_ROOT=Path('/kaggle/working/DermSynReal')
if not (REPO_ROOT/'src/dermsynreal/combine_synthetic.py').is_file():
    subprocess.check_call(['git','clone','--depth','1',
        'https://github.com/shanmukhdatta/DermSynReal.git',str(REPO_ROOT)])
sys.path.insert(0,str(REPO_ROOT/'src'))
from dermsynreal.combine_synthetic import combine_approved
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log


## Edit the mounted path of each reviewed generation manifest and its image root

For each selected class/generator output, one manifest must correspond to the **same saved Kaggle output folder** that contains its generated images. The old `/kaggle/working/...` paths in the saved CSV are not assumed to exist in a new notebook session. For each run, set `image_root` to that run's newly mounted output root.

Add as many entries as you have reviewed and accepted; the six entries are placeholders, not proof those datasets exist. E1 generic, E2 morphology, base and LoRA modes should be combined only with experimentally compatible generator conditions.

In [ ]:
RUNS=[
  {"reviewed_manifest":"/kaggle/input/EDIT_ACNE_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_ACNE_GENERATED_RUN"},
  {"reviewed_manifest":"/kaggle/input/EDIT_FUNGAL_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_FUNGAL_GENERATED_RUN"},
  {"reviewed_manifest":"/kaggle/input/EDIT_VITILIGO_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_VITILIGO_GENERATED_RUN"},
  {"reviewed_manifest":"/kaggle/input/EDIT_HYPERPIGMENTATION_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_HYPERPIGMENTATION_GENERATED_RUN"},
  {"reviewed_manifest":"/kaggle/input/EDIT_SKIN_CANCER_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_SKIN_CANCER_GENERATED_RUN"},
  {"reviewed_manifest":"/kaggle/input/EDIT_NORMAL_REVIEW/reviewed_generation_manifest.csv",
   "image_root":"/kaggle/input/EDIT_NORMAL_GENERATED_RUN"},
]
OUT_DIR="/kaggle/working/dermsynreal_reviewed_synthetic_combination"
OUT_CSV=str(Path(OUT_DIR)/"combined_reviewed_synthetic_pool.csv")
RUN_COMBINE=False
PROMPT_PARTITION=None  # None keeps train/val/test; 'train' for augmentation-only pool
GENERATOR_MODE=None     # e.g. 'base' or 'custom_lora'; use separate runs for differing modes
REQUIRE_ALL_SIX=True
ALLOW_MIXED_GENERATORS=True   # yes for preprocessing, NO for an unmatched E1/E2 experiment arm


In [ ]:
start_notebook_logging(OUT_DIR,kind="synthetic_combine",
    config={"run_count":len(RUNS),"generator_mode_filter":GENERATOR_MODE,
            "prompt_partition_filter":PROMPT_PARTITION},
    repo_root=REPO_ROOT)
if not RUN_COMBINE:
    print("Combine locked. Edit the reviewed mounted CSV/image paths and set RUN_COMBINE=True.")
else:
    result=combine_approved(
        [r["reviewed_manifest"] for r in RUNS],
        [r["image_root"] for r in RUNS],
        OUT_CSV,generator_mode=GENERATOR_MODE,
        prompt_partition=PROMPT_PARTITION,require_six_classes=REQUIRE_ALL_SIX,
        allow_mixed_generators=ALLOW_MIXED_GENERATORS)
    print(json.dumps(result,indent=2))
    append_run_log(OUT_DIR,kind="synthetic_combine_completed",
        config={"approved_images":result["approved_images"],
                "distinct_prompt_families":result["distinct_prompt_families"]},
        artifacts=[OUT_CSV,Path(OUT_CSV).with_suffix(".summary.json")],
        repo_root=REPO_ROOT)
    print("Synthetic preprocessing input CSV:",OUT_CSV)


## Next notebook

Attach the saved `combined_reviewed_synthetic_pool.csv` and the same image-root mounts to `DermaLens_Synthetic_Prompt_Split_Kaggle.ipynb`. Its `INPUT_MODE='manifest'`, `ONLY_HUMAN_ACCEPTED=True` and `PRESERVE_PROMPT_PARTITIONS=True` prevent accidental train-test family mixing. The full synthetic split creates the ZIP and leakage report; classification training then uses its reviewed **train** synthetic rows combined with the unchanged **real** val/test split.

Store `notebook_console.log`, `run_log.jsonl`, combined per-image CSV and generated images in private Kaggle outputs, not public GitHub. Use aggregate CSV exports for the public research evidence folder.